# Import libraries

In [29]:
import os
from re import search
from dfply import *
import scvelo as scv

In [75]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Load settings

In [21]:
if search("ricard", os.uname()[1]):
    exec(open('/Users/ricard/gastrulation_multiome_10x/settings.py').read())
    exec(open('/Users/ricard/gastrulation_multiome_10x/utils.py').read())
elif search("ebi", os.uname()[1]):
    exec(open('/homes/ricard/gastrulation_multiome_10x/settings.py').read())
    exec(open('/homes/ricard/gastrulation_multiome_10x/utils.py').read())
else:
    exit("Computer not recognised")

## Define I/O

In [136]:
io["outfile"] = io["basedir"] + "/processed/rna/velocyto/anndata_scvelo.h5ad"

## Define options 

In [5]:
opts["samples"] = [
	"E7.5_rep1",
	"E7.5_rep2",
	"E8.0_rep1",
	"E8.0_rep2",
	"E8.5_rep1",
	"E8.5_rep2"
]

# Load metadata

In [17]:
metadata = (pd.read_table(io["metadata"]) >>
    mask(X.pass_rnaQC==True, X.doublet_call==False) >>
    mask(X["sample"].isin(opts["samples"]))
).set_index("cell", drop=False)
metadata.shape

(27195, 28)

In [19]:
metadata.index.values

array(['E7.5_rep1_AAACAGCCAAACTCAT-1', 'E7.5_rep1_AAACAGCCACAACCTA-1',
       'E7.5_rep1_AAACAGCCAGGAACTG-1', ...,
       'E8.5_rep2_TTTGTGTTCCTAGTCC-1', 'E8.5_rep2_TTTGTTGGTGCTCCGT-1',
       'E8.5_rep2_TTTGTTGGTGGATTAT-1'], dtype=object)

# Load anndata object

In [23]:
adata = load_adata(adata_file = io["anndata"], metadata_file = io["metadata"], normalise = False, cells = metadata.index.values)

In [24]:
adata.obs.head()

,cell,sample,barcode,archR_cell,nFeature_RNA,nCount_RNA,mtFraction_RNA,pass_rnaQC,celltype.mapped,celltype.score,...,celltype.denoised,TSSEnrichment_atac,ReadsInTSS_atac,PromoterRatio_atac,NucleosomeRatio_atac,nFrags_atac,BlacklistRatio_atac,pass_atacQC,celltype.predicted,stage
cell,,,,,,,,,,,,,,,,,,,,,
E7.5_rep1_AAACAGCCAAACTCAT-1,E7.5_rep1_AAACAGCCAAACTCAT-1,E7.5_rep1,AAACAGCCAAACTCAT-1,E7.5_rep1#AAACAGCCAAACTCAT-1,2929.0,7538.0,16.105068,True,Def._endoderm,0.36,...,Surface_ectoderm,12.593,407.0,0.242666,2.428711,3511.0,0.076616,False,Surface_ectoderm,E7.5
E7.5_rep1_AAACAGCCACAACCTA-1,E7.5_rep1_AAACAGCCACAACCTA-1,E7.5_rep1,AAACAGCCACAACCTA-1,NaN,4125.0,12134.0,20.092303,True,ExE_ectoderm,1.00,...,ExE_ectoderm,NaN,NaN,NaN,NaN,NaN,NaN,False,ExE_ectoderm,E7.5
E7.5_rep1_AAACAGCCAGGAACTG-1,E7.5_rep1_AAACAGCCAGGAACTG-1,E7.5_rep1,AAACAGCCAGGAACTG-1,E7.5_rep1#AAACAGCCAGGAACTG-1,1636.0,3268.0,22.552020,True,Def._endoderm,0.80,...,Def._endoderm,12.400,2173.0,0.175377,1.327706,26061.0,0.024596,True,Def._endoderm,E7.5
E7.5_rep1_AAACAGCCATCCTGAA-1,E7.5_rep1_AAACAGCCATCCTGAA-1,E7.5_rep1,AAACAGCCATCCTGAA-1,E7.5_rep1#AAACAGCCATCCTGAA-1,2676.0,6033.0,21.631029,True,Nascent_mesoderm,0.88,...,Paraxial_mesoderm,9.989,1589.0,0.153302,1.426056,22048.0,0.013176,True,Paraxial_mesoderm,E7.5
E7.5_rep1_AAACAGCCATGCTATG-1,E7.5_rep1_AAACAGCCATGCTATG-1,E7.5_rep1,AAACAGCCATGCTATG-1,E7.5_rep1#AAACAGCCATGCTATG-1,3984.0,11992.0,24.966644,True,Epiblast,0.96,...,Surface_ectoderm,9.207,2771.0,0.166364,1.230269,37415.0,0.016838,True,Surface_ectoderm,E7.5


# Load spliced and unspliced counts from loom files

In [128]:
rename_dict = {
    "E8.5_rep1" : "multiome1",
    "E8.5_rep2" : "multiome2",
    "E8.0_rep1" : "E8_0_rep1_multiome.loom",
    "E8.0_rep2" : "E8_0_rep2_multiome.loom",
    "E7.5_rep1" : "rep1_L001_multiome",
    "E7.5_rep2" : "rep2_L002_multiome"
}

'rep2_L002_multiome'

In [137]:
looms = [None for i in range(len(opts["samples"]))]
for i in range(len(opts["samples"])):
    io["loom_velocyto"] = io["basedir"] + "/processed/rna/velocyto/" + opts["samples"][i] + ".loom"
    looms[i] = sc.read_loom(io["loom_velocyto"], sparse=True, X_name='spliced', obs_names='CellID', obsm_names=None, var_names='Gene')
    looms[i].var_names_make_unique()
    looms[i].obs.index = looms[i].obs.index.str.replace(rename_dict[opts["samples"][i]]+":",opts["samples"][i]+"_").str.replace("x","-1")
    print(looms[i].shape)
    print(looms[i].obs.head())

Variable names are not unique. To make them unique, call `.var_names_make_unique`.


(8933, 55421)
Empty DataFrame
Columns: []
Index: [E7.5_rep1_AAACAGCCATCCTGAA-1, E7.5_rep1_AAACATGCAGGATAAC-1, E7.5_rep1_AAACAGCCATGCTATG-1, E7.5_rep1_AAACCGGCATAATTGC-1, E7.5_rep1_AAAGCCGCACATAGCC-1]


Variable names are not unique. To make them unique, call `.var_names_make_unique`.


(13182, 55421)
Empty DataFrame
Columns: []
Index: [E7.5_rep2_AAACGCGCAACAGGTG-1, E7.5_rep2_AAAGCTTGTAAAGCGG-1, E7.5_rep2_AAACGTACATTGTCCT-1, E7.5_rep2_AAACATGCAAAGCTAA-1, E7.5_rep2_AAACCGGCAGCACGAA-1]


Variable names are not unique. To make them unique, call `.var_names_make_unique`.


(10903, 55421)
Empty DataFrame
Columns: []
Index: [E8.5_rep1_AAACGCGCAATATGGA-1, E8.5_rep1_AAACGCGCACTTCATC-1, E8.5_rep1_AAAGCTTGTTGCAATG-1, E8.5_rep1_AAACATGCAGTCTAGC-1, E8.5_rep1_AAACCAACATTGTGTG-1]


Variable names are not unique. To make them unique, call `.var_names_make_unique`.


(11411, 55421)
Empty DataFrame
Columns: []
Index: [E8.5_rep2_AAACGTACATGTGGGA-1, E8.5_rep2_AAAGGAGCAACAGGAT-1, E8.5_rep2_AAACAGCCAAGGACCA-1, E8.5_rep2_AAACCGGCAATAATGG-1, E8.5_rep2_AAAGCCGCAAACCTTG-1]


In [151]:
adata_loom = anndata.AnnData.concatenate(*looms, join='inner', batch_key=None, index_unique=None)
del looms

In [157]:
adata.shape
adata_loom.shape

(27195, 32285)

(44429, 55421)

Remove non-used layers to save memory

In [154]:
del adata_loom.layers["ambiguous"]
del adata_loom.layers["matrix"]

Merge anndata objects

In [161]:
adata_final = scv.utils.merge(adata, adata_loom)
del adata_loom
del adata
adata_final

In [169]:
adata_final.obs.index.name = None

In [171]:
adata_final.obs.head()

,cell,sample,barcode,archR_cell,nFeature_RNA,nCount_RNA,mtFraction_RNA,pass_rnaQC,celltype.mapped,celltype.score,...,PromoterRatio_atac,NucleosomeRatio_atac,nFrags_atac,BlacklistRatio_atac,pass_atacQC,celltype.predicted,stage,initial_size_spliced,initial_size_unspliced,initial_size
E7.5_rep1_AAACAGCCAAACTCAT-1,E7.5_rep1_AAACAGCCAAACTCAT-1,E7.5_rep1,AAACAGCCAAACTCAT-1,E7.5_rep1#AAACAGCCAAACTCAT-1,2929.0,7538.0,16.105068,True,Def._endoderm,0.36,...,0.242666,2.428711,3511.0,0.076616,False,Surface_ectoderm,E7.5,6130,637,6130.0
E7.5_rep1_AAACAGCCACAACCTA-1,E7.5_rep1_AAACAGCCACAACCTA-1,E7.5_rep1,AAACAGCCACAACCTA-1,nan,4125.0,12134.0,20.092303,True,ExE_ectoderm,1.00,...,NaN,NaN,NaN,NaN,False,ExE_ectoderm,E7.5,7147,3678,7147.0
E7.5_rep1_AAACAGCCAGGAACTG-1,E7.5_rep1_AAACAGCCAGGAACTG-1,E7.5_rep1,AAACAGCCAGGAACTG-1,E7.5_rep1#AAACAGCCAGGAACTG-1,1636.0,3268.0,22.552020,True,Def._endoderm,0.80,...,0.175377,1.327706,26061.0,0.024596,True,Def._endoderm,E7.5,2022,891,2022.0
E7.5_rep1_AAACAGCCATCCTGAA-1,E7.5_rep1_AAACAGCCATCCTGAA-1,E7.5_rep1,AAACAGCCATCCTGAA-1,E7.5_rep1#AAACAGCCATCCTGAA-1,2676.0,6033.0,21.631029,True,Nascent_mesoderm,0.88,...,0.153302,1.426056,22048.0,0.013176,True,Paraxial_mesoderm,E7.5,3349,2018,3349.0
E7.5_rep1_AAACAGCCATGCTATG-1,E7.5_rep1_AAACAGCCATGCTATG-1,E7.5_rep1,AAACAGCCATGCTATG-1,E7.5_rep1#AAACAGCCATGCTATG-1,3984.0,11992.0,24.966644,True,Epiblast,0.96,...,0.166364,1.230269,37415.0,0.016838,True,Surface_ectoderm,E7.5,7554,3098,7554.0


# Save anndata object

In [172]:
adata_final.write_h5ad(io["outfile"])